# Statistical analysis
Statistics, tables and figures for *Training stability and architecture comparison in deep learning-based lung CT segmentation*.

Data: COVID-19-CT-Seg (Zenodo record 3757476), the ten `coronacases` volumes in Hounsfield units, 267 lung slices.

Reads the per-run results in `results/runs/` and writes the full report and tables to `results/tables/`, the article figures to `results/figures/` and supplementary figures to `results/figures/supplementary/`. Run from the `notebooks/` folder.

## Setup

In [ ]:
from __future__ import annotations
import glob
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import json, tifffile
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9, 'axes.grid': True, 'grid.alpha': .3})
C = {'unet': '#4c72b0', 'attn_unet': '#dd8452', 'resunet': '#55a868', 'resunetpp': '#c44e52'}

def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(FIG, f'{name}.{ext}'), bbox_inches='tight', dpi=300)
    plt.show()

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

BASE = os.path.normpath(os.path.join(os.getcwd(), ".."))          # run from notebooks/
RES = os.path.join(BASE, "results", "runs")                      # written by the training scripts
IDX = os.path.join(BASE, "data", "processed", "dataset_index.csv")
TAB = os.path.join(BASE, "results", "tables")
FIG = os.path.join(BASE, "results", "figures", "supplementary")
os.makedirs(FIG, exist_ok=True)
os.makedirs(TAB, exist_ok=True)
RNG = np.random.default_rng(20260908)            # bootstrap resampling and plot jitter
TOST_MARGIN, SPLIT_MARGIN = 0.02, 0.03

_lines: list[str] = []
def P(s=""):
    print(s); _lines.append(str(s))


def boot_ci(x, fn=np.mean, n=10000, a=0.05):
    x = np.asarray(x, float)
    bs = [fn(RNG.choice(x, x.size, replace=True)) for _ in range(n)]
    return float(fn(x)), float(np.percentile(bs, 100*a/2)), float(np.percentile(bs, 100*(1-a/2)))


def paired_boot_ci(d, n=10000, a=0.05):
    return boot_ci(d, np.mean, n, a)


def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return max(0, c-h), min(1, c+h)


def holm(pvals):
    p = np.asarray(pvals, float); order = np.argsort(p); m = len(p)
    adj = np.empty(m); run = 0.0
    for i, idx in enumerate(order):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(run, 1.0)
    return adj


P("="*78); P("STATISTICAL ANALYSIS -- COVID-19-CT-Seg (10 patients, 267 slices)"); P("="*78)

idx_all = pd.read_csv(IDX)
print(len(idx_all), 'slices,', idx_all.patient_id.nunique(), 'patients')

## A. Baseline: seed variability
Sections 4.2–4.3 and 5: ResUNet++ baseline with gradient accumulation, 20 seeds. The literal-batch-10 baseline (8 seeds) is analysed in section G.

In [ ]:
P("\n\n########## A. Baseline (gradient accumulation): seed variability ##########")
acc = pd.read_csv(f"{RES}/baseline_accumulation.csv")
m, lo, hi = boot_ci(acc.heldout_dice)
kc = int((acc.heldout_dice < 0.80).sum()); wl, wh = wilson(kc, len(acc))
P(f"\nGradient-accumulation engine (batch 10 = micro-batches of 4 + 4 + 2), n={len(acc)} seeds")
P(f"  held-out Dice@0.5 : mean {m:.4f}  95%CI [{lo:.4f}, {hi:.4f}]  SD {acc.heldout_dice.std():.4f}")
P(f"  range [{acc.heldout_dice.min():.4f}, {acc.heldout_dice.max():.4f}]  median {acc.heldout_dice.median():.4f}")
P(f"  collapses (Dice<0.80): {kc}/{len(acc)}  (Wilson 95%CI [{wl:.2f}, {wh:.2f}])")
P(f"  held-out soft Dice : mean {acc.heldout_soft_dice.mean():.4f}  SD {acc.heldout_soft_dice.std():.4f}")
P(f"  held-out accuracy  : mean {acc.heldout_acc.mean():.4f}  SD {acc.heldout_acc.std():.4f}")
P(f"  final-epoch training loss: mean {acc.final_train_loss.mean():.4f}")
P("  Literal batch 10 (8 seeds): see section G.")
pd.DataFrame({"quantity": ["n", "dice_mean", "ci_lo", "ci_hi", "dice_sd", "dice_min", "dice_max",
                           "collapse_k", "soft_dice_mean", "acc_mean"],
              "value": [len(acc), m, lo, hi, acc.heldout_dice.std(), acc.heldout_dice.min(),
                        acc.heldout_dice.max(), kc, acc.heldout_soft_dice.mean(), acc.heldout_acc.mean()]}
             ).to_csv(f"{TAB}/table_A_baseline.csv", index=False)

In [ ]:
# Supplementary figure: validation loss and soft Dice per epoch (baseline, 20 seeds)
H = json.load(open(f"{RES}/baseline_accumulation_histories.json"))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for s, h in sorted(H.items(), key=lambda kv: int(kv[0])):
    e = [r["epoch"] for r in h]
    ax[0].plot(e, [r["val_loss"] for r in h], lw=.8, alpha=.7)
    ax[1].plot(e, [r["val_soft_dice"] for r in h], lw=.8, alpha=.7)
ax[0].set_yscale("log"); ax[0].set(xlabel="epoch", ylabel="validation loss (log)", title="A  Validation loss, baseline (20 seeds)")
ax[1].set(xlabel="epoch", ylabel="validation soft Dice", ylim=(0, 1), title="B  Validation soft Dice")
save(fig, "supp_baseline_curves")

# Figure: network-input distribution, /255 vs lung window (24 slices pooled)
ids = idx_all.sort_values("img_id").iloc[::11]["image_path"][:24]
raw = np.concatenate([tifffile.imread(p).ravel() for p in ids]).astype(np.float32)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
x1 = raw / 255.0
ax[0].hist(x1, bins=200, color="#c44e52"); ax[0].set_yscale("log")
ax[0].set(title=f"A  input after image/255  range [{x1.min():.1f}, {x1.max():.1f}]", xlabel="network input value", ylabel="pixels")
x2 = (np.clip(raw, -1350, 150) + 1350) / 1500
ax[1].hist(x2, bins=200, color="#4c72b0"); ax[1].set_yscale("log")
ax[1].set(title="B  input after lung-window scaling  [0, 1]", xlabel="network input value")
save(fig, "supp_input_distribution")

## B. Progressive protocol (Sections 4.4 and 5.1, Table 4)
Four cumulative arms on 12 matched seeds.

In [ ]:
P("\n\n########## B. Progressive protocol (4 arms, seed-paired) ##########")
gd = pd.read_csv(f"{RES}/progressive_protocol.csv")
arms = ["A_baseline", "B_best_epoch_threshold", "C_lung_window", "D_augmentation"]
rows = []
for a in arms:
    s = gd.loc[gd.arm == a, "heldout_dice"]
    mm, l, h = boot_ci(s)
    rows.append(dict(arm=a, n=len(s), dice_mean=mm, dice_median=s.median(), ci_lo=l, ci_hi=h,
                     dice_sd=s.std(), dice_min=s.min(), collapse_lt80=int((s < 0.80).sum())))
arm_table = pd.DataFrame(rows)
P("\nPer-arm held-out Dice:")
for r in arm_table.itertuples():
    P(f"  {r.arm:24s} mean {r.dice_mean:.4f}  median {r.dice_median:.4f}  95%CI [{r.ci_lo:.4f},{r.ci_hi:.4f}]  "
      f"SD {r.dice_sd:.4f}  min {r.dice_min:.4f}  collapse {r.collapse_lt80}/{r.n}")

P("\nPaired step deltas (same seed; Wilcoxon signed-rank + paired bootstrap 95% CI, Holm):")
piv = gd.pivot_table(index="seed", columns="arm", values="heldout_dice")
steps = [("A_baseline", "B_best_epoch_threshold"), ("B_best_epoch_threshold", "C_lung_window"),
         ("C_lung_window", "D_augmentation"), ("A_baseline", "D_augmentation")]
brows = []
for a, b in steps:
    d = (piv[b] - piv[a]).dropna()
    md, l, h = paired_boot_ci(d.values)
    pv = stats.wilcoxon(piv[b].values, piv[a].values).pvalue
    brows.append(dict(step=f"{a}->{b}", delta_mean=md, ci_lo=l, ci_hi=h, wilcoxon_p=pv,
                      sd_before=piv[a].std(), sd_after=piv[b].std(), n_improved=int((d > 0).sum()), n=len(d)))
step_table = pd.DataFrame(brows)
step_table["holm_p"] = np.r_[holm(step_table.wilcoxon_p[:3]), np.nan]   # Holm over the 3 consecutive steps
for r in step_table.itertuples():
    hp = "" if np.isnan(r.holm_p) else f" (Holm {r.holm_p:.3g})"
    P(f"  {r.step:47s} Δ {r.delta_mean:+.4f}  95%CI [{r.ci_lo:+.4f},{r.ci_hi:+.4f}]  "
      f"p={r.wilcoxon_p:.3g}{hp}  improved {r.n_improved}/{r.n}  SD {r.sd_before:.4f}->{r.sd_after:.4f}")

P("\nAcross-seed variability (Levene, median-centred) and SD ratio with bootstrap 95% CI:")
vrows = []
for a, b in [("A_baseline", "B_best_epoch_threshold"), ("A_baseline", "C_lung_window"), ("B_best_epoch_threshold", "C_lung_window")]:
    va, vb = piv[a].dropna().values, piv[b].dropna().values
    rr = [RNG.choice(va, va.size).std(ddof=1) / max(RNG.choice(vb, vb.size).std(ddof=1), 1e-9)
          for _ in range(10000)]
    lev = stats.levene(va, vb, center="median")
    ratio = va.std(ddof=1) / vb.std(ddof=1)
    P(f"  {a} vs {b}: SD {va.std(ddof=1):.4f} vs {vb.std(ddof=1):.4f}  ratio {ratio:.2f}x "
      f"[{np.percentile(rr, 2.5):.2f}, {np.percentile(rr, 97.5):.2f}]  Levene p={lev.pvalue:.3g}")
    vrows.append(dict(pair=f"{a} vs {b}", sd_a=va.std(ddof=1), sd_b=vb.std(ddof=1), ratio=ratio,
                      ratio_ci_lo=np.percentile(rr, 2.5), ratio_ci_hi=np.percentile(rr, 97.5), levene_p=lev.pvalue))
arm_table.to_csv(f"{TAB}/table_B_progressive_arms.csv", index=False)
step_table.to_csv(f"{TAB}/table_B_progressive_steps.csv", index=False)
pd.DataFrame(vrows).to_csv(f"{TAB}/table_B_progressive_variance.csv", index=False)

In [ ]:
# Supplementary figure: progressive protocol and distribution of run outcomes
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"wspace": .35})
lab = ["A\n/255, last-epoch", "B\n+best-epoch\n+tuned thr", "C\n+lung window", "D\n+augment"]
for i, a in enumerate(arms):
    v = gd.loc[gd.arm == a, "heldout_dice"].values
    ax[0].scatter(np.full(v.size, i) + RNG.uniform(-.12, .12, v.size), v, s=18, alpha=.75)
    ax[0].hlines(v.mean(), i-.25, i+.25, color="k", lw=2)
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(lab, fontsize=8)
ax[0].set(ylabel="held-out Dice @thr (12 seeds)", title="A  Progressive protocol")
groups = [("Baseline, accumulation (20)", acc.heldout_dice.values)] + [(f"Arm {a[0]} (12)", gd.loc[gd.arm == a, "heldout_dice"].values) for a in arms]
for i, (g, v) in enumerate(groups):
    ax[1].scatter(v, np.full(v.size, len(groups)-1-i) + RNG.uniform(-.15, .15, v.size), s=14, alpha=.7)
    ax[1].vlines(v.mean(), len(groups)-1-i-.3, len(groups)-1-i+.3, color="k", lw=2)
ax[1].set_yticks(range(len(groups))); ax[1].set_yticklabels([g for g, _ in groups][::-1])
ax[1].set(xlabel="held-out Dice @0.5", title="B  Distribution of run outcomes")
save(fig, "supp_progressive_protocol")

## C. Architecture comparison and equivalence (Sections 4.5 and 5.2, Table 5)
Friedman test with Nemenyi post-hoc on per-image Dice, pairwise paired bootstrap, TOST at ±0.02 on the 90% CI.

In [ ]:
P("\n\n########## C. Architecture comparison and equivalence ##########")
pid = pd.concat([pd.read_csv(f) for f in glob.glob(f"{RES}/per_image_dice/*_seed*.csv")], ignore_index=True)
pim = pid.groupby(["arch", "img_id"]).dice.mean().reset_index()
wide = pim.pivot(index="img_id", columns="arch", values="dice").dropna()
archs = sorted(wide.columns)
ab = pd.read_csv(f"{RES}/architecture_benchmark.csv")
rm = ab.groupby("arch").agg(dice=("heldout_dice", "mean"), sd=("heldout_dice", "std"), iou=("heldout_iou", "mean"),
                            acc=("heldout_acc", "mean"), ece=("heldout_ece", "mean"), secs=("secs", "mean"))
P(f"\nn images (held out at least once in every architecture) = {len(wide)} ; architectures = {archs}")
P("\nRun-level (5 seeds each):")
for a in archs:
    r = rm.loc[a]
    P(f"  {a:12s} Dice {r.dice:.4f} +/- {r.sd:.4f}  IoU {r.iou:.4f}  acc {r.acc:.4f}  ECE {r.ece:.4f}  {r.secs:.0f} s/run")
fr = stats.friedmanchisquare(*[wide[a].values for a in archs])
ranks = wide[archs].rank(axis=1, ascending=False).mean()
k, N = len(archs), len(wide)
cd = 2.569 * np.sqrt(k*(k+1)/(6*N))            # Nemenyi q_0.05 for k=4
P(f"\nFriedman (per-image, k={k}, n={N}): chi2={fr.statistic:.3f}  p={fr.pvalue:.4g}")
P("  mean ranks: " + "  ".join(f"{a} {ranks[a]:.2f}" for a in archs) + f"   Nemenyi CD = {cd:.3f}")
P(f"\nPairwise: paired bootstrap Δ (per-image Dice), Wilcoxon (Holm), TOST at ±{TOST_MARGIN} on the 90% CI:")
prs = []
for i in range(k):
    for j in range(i+1, k):
        a, b = archs[i], archs[j]
        d = (wide[a] - wide[b]).values
        md, l90, h90 = paired_boot_ci(d, a=0.10)
        _, l95, h95 = paired_boot_ci(d, a=0.05)
        prs.append(dict(pair=f"{a} vs {b}", delta=md, ci90_lo=l90, ci90_hi=h90, ci95_lo=l95, ci95_hi=h95,
                        wilcoxon_p=stats.wilcoxon(wide[a].values, wide[b].values).pvalue,
                        rank_diff=abs(ranks[a]-ranks[b]),
                        tost_equivalent=bool(l90 > -TOST_MARGIN and h90 < TOST_MARGIN)))
pair_table = pd.DataFrame(prs); pair_table["holm_p"] = holm(pair_table.wilcoxon_p); pair_table["nemenyi_sig"] = pair_table.rank_diff > cd
for r in pair_table.itertuples():
    P(f"  {r.pair:24s} Δ {r.delta:+.4f}  90%CI [{r.ci90_lo:+.4f},{r.ci90_hi:+.4f}]  p={r.wilcoxon_p:.3g} "
      f"(Holm {r.holm_p:.3g})  Nemenyi sig: {r.nemenyi_sig}  TOST equiv: {r.tost_equivalent}")
P(f"\n  All pairs TOST-equivalent at ±{TOST_MARGIN}: {bool(pair_table.tost_equivalent.all())}")
P(f"  max |Δ| = {pair_table.delta.abs().max():.4f};  ResUNet++ / U-Net time ratio = {rm.loc['resunetpp','secs']/rm.loc['unet','secs']:.2f}x")
pair_table.to_csv(f"{TAB}/table_C_pairwise.csv", index=False)
rm.assign(mean_rank=ranks).to_csv(f"{TAB}/table_C_architectures.csv")

In [ ]:
# Figure: per-image Dice by architecture with equivalence band + critical-difference diagram
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
ax[0].boxplot([wide[a].values for a in archs], showfliers=True, flierprops=dict(ms=3))
gm = np.median(wide.values)   # median: a few near-empty edge slices drag the mean to ~0.95
ax[0].axhspan(gm - TOST_MARGIN, gm + TOST_MARGIN, color="grey", alpha=.2, label="±0.02 margin around median (scale only)")
ax[0].set_xticks(range(1, k+1)); ax[0].set_xticklabels(archs)
ax[0].set(ylabel="per-image Dice (mean over 5 seeds)", title=f"A  Architectures (n={N} images)"); ax[0].legend(fontsize=7, loc="lower left")
ax[1].set_xlim(1, k); ax[1].set_ylim(-1, len(archs)); ax[1].invert_xaxis(); ax[1].grid(False); ax[1].set_yticks([])
for i, a in enumerate(ranks.sort_values().index):
    ax[1].plot(ranks[a], 0, "o", color=C[a]); ax[1].text(ranks[a], i + .4, f"{a}\n({ranks[a]:.2f})", ha="center", fontsize=8)
    ax[1].vlines(ranks[a], 0, i + .35, color=C[a], lw=.8)
ax[1].hlines(-.5, ranks.min(), ranks.min() + cd, color="k", lw=3); ax[1].text(ranks.min() + cd/2, -.9, f"CD = {cd:.2f}", ha="center")
ax[1].set(xlabel="mean rank (Friedman + Nemenyi)", title="B  Critical-difference diagram")
save(fig, "supp_architectures")

## D. Residual-error regression (Section 5.3, Table 6)
Fractional-response GLM (logit) of per-image Dice on lung area fraction, mean lung attenuation and distance from the middle of the lung (slice position normalised within each volume). Standard errors clustered by image (Table 6) and, as a sensitivity analysis, by patient.

In [ ]:
P("\n\n########## D. Residual-error regression (per-image Dice ~ covariates) ##########")
idx = pd.read_csv(IDX)
df = pid.merge(idx[["img_id", "patient_id", "z_norm", "lung_volume_fraction", "lung_mean_hu"]], on="img_id")
df["abs_z_dev"] = (df.z_norm - 0.5).abs()
covs = ["lung_volume_fraction", "lung_mean_hu", "abs_z_dev"]
for c in covs:
    df[c+"_z"] = (df[c] - df[c].mean()) / df[c].std()
df["_y"] = df.dice.clip(1e-4, 1-1e-4)
terms = [c+"_z" for c in covs]
drows = []
for label, groups in [("image", df.img_id), ("patient", df.patient_id)]:
    fit = smf.glm("_y ~ " + " + ".join(terms), data=df, family=sm.families.Binomial()).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
    ci = fit.conf_int(); adj = holm([fit.pvalues[t] for t in terms])
    P(f"\nFractional-response GLM (logit), cluster-robust by {label}; n_obs={len(df)}, "
      f"n_images={df.img_id.nunique()}, n_patients={df.patient_id.nunique()}:")
    for t, ap in zip(terms, adj):
        P(f"  {t:26s} beta {fit.params[t]:+.3f}  95%CI [{ci.loc[t,0]:+.3f},{ci.loc[t,1]:+.3f}]  "
          f"p={fit.pvalues[t]:.3g} (Holm {ap:.3g})")
        drows.append(dict(cluster=label, term=t, beta=fit.params[t], ci_lo=ci.loc[t, 0], ci_hi=ci.loc[t, 1],
                          p=fit.pvalues[t], holm_p=ap))
pd.DataFrame(drows).to_csv(f"{TAB}/table_D_regression.csv", index=False)
P("\nTertile means of image-level Dice (averaged over archs + seeds):")
img = df.groupby("img_id").agg(dice=("dice", "mean"), **{c: (c, "first") for c in covs})
trows = []
for c in covs:
    q = pd.qcut(img[c], 3, labels=["low", "mid", "high"])
    g = img.groupby(q, observed=True).dice.agg(["mean", "count"])
    P(f"  {c}: " + "  ".join(f"{lv} {r['mean']:.4f} (n={int(r['count'])})" for lv, r in g.iterrows()))
    trows += [dict(covariate=c, tertile=lv, mean_dice=r["mean"], n=int(r["count"])) for lv, r in g.iterrows()]
pd.DataFrame(trows).to_csv(f"{TAB}/table_D_tertiles.csv", index=False)
pp = df.groupby("patient_id").dice.agg(["mean", "min", "count"])
P("\nPer-patient mean held-out Dice (architecture benchmark, pooled):")
for p_, r in pp.iterrows():
    P(f"  {p_:18s} mean {r['mean']:.4f}  min {r['min']:.4f}  (obs {int(r['count'])})")
pp.to_csv(f"{TAB}/table_D_per_patient.csv")

In [ ]:
# Figure: residual per-image Dice vs covariates (red = binned mean) + per-patient Dice
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4), gridspec_kw={"wspace": .3})
for a_, c, lbl in zip(ax[:3], covs, ["lung area fraction", "mean lung HU", "|z_norm - 0.5| (within-patient)"]):
    a_.scatter(img[c], img.dice, s=8, alpha=.5)
    b = pd.qcut(img[c], 6); bm = img.groupby(b, observed=True).agg(x=(c, "mean"), y=("dice", "mean"))
    a_.plot(bm.x, bm.y, "r-o", ms=3); a_.set(xlabel=lbl, ylim=(-.02, 1.02))
ax[0].set_ylabel("per-image Dice")
ax[3].bar(range(len(pp)), pp["mean"], color="#4c72b0")
ax[3].set_xticks(range(len(pp))); ax[3].set_xticklabels([p[-2:] for p in pp.index], fontsize=8)
ax[3].set(ylim=(.85, 1), xlabel="patient (coronacases_0##)", ylabel="mean per-image Dice", title="per patient (benchmark, pooled)")
save(fig, "supp_residual_error")

# Figure: worst per-image cases -- how many near-empty predictions sit at lung apex/base
worst = img.sort_values("dice").head(10).join(idx_all.set_index("img_id")[["patient_id", "z_norm", "lung_area_px"]])
worst

## E. Qualitative examples (Figure 12)
One U-Net trained under the stabilized protocol (seed 0) by `scripts/make_qualitative.py`. Its test predictions are shown for a typical slice, the densest slice (highest mean lung HU) and the worst slice whose reference mask covers at least 2% of the image. Reference in green, prediction in red.

In [ ]:
QNPZ = os.path.join(RES, "qualitative", "unet_seed0.npz")   # written by scripts/make_qualitative.py
if not os.path.exists(QNPZ):
    print("no", QNPZ, "-> qualitative figure (Figure 12) skipped")
else:
    q = np.load(QNPZ, allow_pickle=True)
    qid, qimg, qgt, qpr, thr = [str(i) for i in q["ids"]], q["img"], q["gt"] > .5, q["pred"] > float(q["thr"]), float(q["thr"])
    qd = np.array([2*(p & g).sum() / (p.sum() + g.sum() + 1e-7) for p, g in zip(qpr, qgt)])
    meta = idx_all.set_index("img_id").loc[qid].reset_index()
    meta["dice"], meta["gt_frac"] = qd, qgt.reshape(len(qid), -1).mean(1)
    ok = meta[meta.gt_frac >= .02]
    pick = [("typical", (ok.dice - ok.dice.median()).abs().idxmin()),
            ("densest lung", ok.lung_mean_hu.idxmax()),
            ("worst (well-formed mask)", ok.dice.idxmin())]
    fig, ax = plt.subplots(len(pick), 3, figsize=(9, 3 * len(pick)))
    for r, (name, i) in enumerate(pick):
        im = np.clip(qimg[i], -1350, 150)
        for c, (ttl, show) in enumerate([("CT (lung window)", None), ("ground truth", qgt[i]), ("prediction", qpr[i])]):
            a = ax[r, c]; a.imshow(im, cmap="gray"); a.axis("off")
            if c >= 1: a.contour(qgt[i], [.5], colors="lime", linewidths=.8, linestyles=":" if c == 2 else "-")
            if c == 2: a.contour(qpr[i], [.5], colors="red", linewidths=.8)
            a.set_title(ttl if r == 0 else "", fontsize=9)
        m = meta.loc[i]
        ax[r, 0].text(-.05, .5, f"{name}\n{m.patient_id[-3:]} z={m.z}\nHU {m.lung_mean_hu:.0f}\nDice {m.dice:.3f}",
                      transform=ax[r, 0].transAxes, ha="right", va="center", fontsize=8)
    save(fig, "supp_qualitative")
    print(f"threshold {thr:.2f}; held-out slices {len(qid)}; mean Dice {qd.mean():.4f}; slices with empty prediction: {(qpr.reshape(len(qid), -1).sum(1) == 0).sum()}")
    meta.sort_values("dice")[["img_id", "patient_id", "z_norm", "gt_frac", "lung_mean_hu", "dice"]].head(8)

## F. Split sensitivity (Sections 4.6 and 5.4)
Random slice split vs Z-blocked split (paired by seed; each of the four blocks is held out once per seed), and leave-one-patient-out (unpaired).

In [ ]:
P("\n\n########## F. Split sensitivity: random vs Z-blocked vs leave-one-patient-out ##########")
lp = pd.read_csv(f"{RES}/split_sensitivity.csv").rename(columns={"split": "scheme"})
# Z-blocked: each block is held out once per seed -> one value per seed (mean over the folds)
zb_all = lp[lp.scheme == "z_blocked"]
P(f"Z-blocked: {zb_all.z_fold.nunique()} folds per seed, averaged per seed")
zb_mean = zb_all.groupby(["arch", "scheme", "seed"], as_index=False)[["heldout_dice"]].mean()
lp = pd.concat([lp[lp.scheme != "z_blocked"], zb_mean], ignore_index=True)
P("\nPer (arch, scheme) held-out Dice:")
srows = []
for (a, sch), s in lp.groupby(["arch", "scheme"]).heldout_dice:
    mm, l, h = boot_ci(s.values)
    P(f"  {a:10s} {sch:13s} mean {mm:.4f}  95%CI [{l:.4f},{h:.4f}]  SD {s.std():.4f}  min {s.min():.4f}  n={len(s)}")
    srows.append(dict(arch=a, scheme=sch, n=len(s), mean=mm, ci_lo=l, ci_hi=h, sd=s.std(), min=s.min()))
pd.DataFrame(srows).to_csv(f"{TAB}/table_F_split_schemes.csv", index=False)

P(f"\nContrasts (random_slice minus other scheme; pre-specified practical margin {SPLIT_MARGIN}):")
e_rows = []
for a in sorted(lp.arch.unique()):
    sub = lp[lp.arch == a]
    rnd = sub[sub.scheme == "random_slice"].set_index("seed").heldout_dice
    zb = sub[sub.scheme == "z_blocked"].set_index("seed").heldout_dice
    lo_ = sub[sub.scheme == "patient_lopo"].heldout_dice.values
    d = (rnd - zb).dropna()
    md, l, h = paired_boot_ci(d.values)
    pz = stats.wilcoxon(rnd.loc[d.index], zb.loc[d.index]).pvalue
    P(f"  {a:10s} random - z_blocked    Δ {md:+.4f}  95%CI [{l:+.4f},{h:+.4f}]  paired Wilcoxon p={pz:.3g}  "
      f"within margin: {bool(l > -SPLIT_MARGIN and h < SPLIT_MARGIN)}")
    e_rows.append(dict(arch=a, contrast="random - z_blocked", paired=True, delta=md, ci_lo=l, ci_hi=h, p=pz,
                       within_margin=bool(l > -SPLIT_MARGIN and h < SPLIT_MARGIN)))
    bs = [RNG.choice(rnd.values, rnd.size).mean() - RNG.choice(lo_, lo_.size).mean() for _ in range(10000)]
    md2 = rnd.mean() - lo_.mean(); l2, h2 = np.percentile(bs, [2.5, 97.5])
    pm = stats.mannwhitneyu(rnd.values, lo_).pvalue
    lev = stats.levene(rnd.values, lo_, center="median").pvalue
    P(f"  {a:10s} random - patient_lopo Δ {md2:+.4f}  95%CI [{l2:+.4f},{h2:+.4f}]  Mann-Whitney p={pm:.3g}  "
      f"SD {rnd.std():.4f} vs {lo_.std():.4f} (x{lo_.std()/rnd.std():.1f}, Levene p={lev:.3g})  "
      f"within margin: {bool(l2 > -SPLIT_MARGIN and h2 < SPLIT_MARGIN)}")
    e_rows.append(dict(arch=a, contrast="random - patient_lopo", paired=False, delta=md2, ci_lo=l2, ci_hi=h2, p=pm,
                       sd_random=rnd.std(), sd_lopo=lo_.std(), levene_p=lev,
                       within_margin=bool(l2 > -SPLIT_MARGIN and h2 < SPLIT_MARGIN)))
pd.DataFrame(e_rows).to_csv(f"{TAB}/table_F_split_contrasts.csv", index=False)

lopo = lp[lp.scheme == "patient_lopo"].assign(patient=lambda x: x.test_patients)
P("\nLOPO per held-out patient (Dice):")
w = lopo.pivot_table(index="patient", columns="arch", values="heldout_dice")
for p_, r in w.iterrows():
    P(f"  {p_:18s} " + "  ".join(f"{a} {r[a]:.4f}" for a in w.columns))
w.to_csv(f"{TAB}/table_F_lopo_per_patient.csv")

In [ ]:
# Figure: split sensitivity, incl. leave-one-patient-out, and LOPO Dice per held-out patient
order = ["random_slice", "z_blocked", "patient_lopo"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for j, a in enumerate(["unet", "resunetpp"]):
    for i, sch in enumerate(order):
        v = lp[(lp.arch == a) & (lp.scheme == sch)].heldout_dice.values
        x = i + (j - .5) * .35
        ax[0].scatter(np.full(v.size, x) + RNG.uniform(-.06, .06, v.size), v, s=16, color=C[a], alpha=.75,
                      label=a if i == 0 else None)
        ax[0].hlines(v.mean(), x - .12, x + .12, color="k", lw=2)
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(["random slice\n(6 seeds)", "Z-blocked K=4\n(6 seeds x 4 folds)", "leave-one-patient-out\n(10 folds)"])
ax[0].set(ylabel="held-out Dice @thr", title="A  Split sensitivity"); ax[0].legend(fontsize=8)
xx = np.arange(len(w))
for j, a in enumerate(["unet", "resunetpp"]):
    ax[1].bar(xx + (j - .5) * .38, w[a], width=.38, color=C[a], label=a)
ax[1].set_xticks(xx); ax[1].set_xticklabels([p[-3:] for p in w.index])
ax[1].set(ylim=(.94, 1), xlabel="held-out patient (coronacases_###)", ylabel="Dice", title="B  LOPO: Dice per held-out patient")
ax[1].legend(fontsize=8)
save(fig, "supp_split_sensitivity")

## G. Article figures and literal-batch-10 baseline
Writes Figures 4–13 to `results/figures/`, named by article figure number, and prints the literal-batch-10 statistics.

In [ ]:
# Article figures, numbered as in the article
PF = os.path.join(BASE, "results", "figures")
os.makedirs(PF, exist_ok=True)
def psave(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(PF, f"{name}.{ext}"), bbox_inches="tight", dpi=300)
    plt.show()

# Figure 4 + Figure 5: literal batch 10 (only once the 8-seed run exists)
TB = RES
tb_csv = os.path.join(TB, "literal_batch10.csv")
if os.path.exists(tb_csv):
    tb = pd.read_csv(tb_csv).sort_values("seed")
    hist10 = {s: pd.read_csv(os.path.join(TB, f"literal_batch10_history_seed{s}.csv")) for s in tb.seed}
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for s, h in hist10.items():
        ax[0].plot(h.epoch, h.val_loss, label=f"seed {s}")
        ax[1].plot(h.epoch, h.val_dice)
    ax[0].set_yscale("log")
    ax[0].set(xlabel="epoch", ylabel="validation loss (log)", title="A  Validation loss, literal batch 10")
    ax[0].legend(fontsize=7, ncol=2)
    ax[1].set(xlabel="epoch", ylabel="validation Dice @0.5", ylim=(0, 1), title="B  Validation Dice, literal batch 10")
    psave(fig, "fig4_tb10_curves")

    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.scatter(tb.heldout_dice, tb.heldout_dice_best_epoch, s=30)
    for r in tb.itertuples():
        ax.annotate(f"s{r.seed}", (r.heldout_dice, r.heldout_dice_best_epoch), fontsize=7,
                    xytext=(3, 2), textcoords="offset points")
    lo = min(tb.heldout_dice.min(), tb.heldout_dice_best_epoch.min()) - .01
    ax.plot([lo, 1], [lo, 1], "k:", lw=.8)
    ax.set(xlabel="held-out Dice, final epoch", ylabel="held-out Dice, best validation epoch",
           title=f"Literal batch 10: final epoch vs best epoch\n({len(tb)} seeds)")
    psave(fig, "fig5_best_vs_final")

    f10, b10 = tb.heldout_dice, tb.heldout_dice_best_epoch
    m10, l10, h10 = boot_ci(f10)
    k10 = int((f10 < .8).sum()); w10 = wilson(k10, len(tb))
    spikes = sum((h.val_loss[h.epoch <= 10] > 1).any() for h in hist10.values())
    learned = {s: h for s, h in hist10.items() if (h.val_dice > .5).any()}
    exc = sum((h.loc[(h.val_dice > .5).idxmax():, "val_loss"] > .3).any() for h in learned.values())
    print(f"literal batch 10 (n={len(tb)}): final {m10:.4f} +/- {f10.std():.4f}  95%CI [{l10:.3f}, {h10:.3f}]  "
          f"range [{f10.min():.4f}, {f10.max():.4f}]")
    print(f"  collapses {k10}/{len(tb)} (Wilson {w10[0]:.2f}-{w10[1]:.2f}); best epoch {b10.mean():.4f} +/- {b10.std():.4f} "
          f"min {b10.min():.4f}; best-final mean {(b10 - f10).mean():.4f} max {(b10 - f10).max():.4f}")
    print(f"  SD ratio batch10 / accumulation = {f10.std() / acc.heldout_dice.std():.1f}x; "
          f"val-loss spikes > 1 in the first 10 epochs: {spikes}/{len(tb)} seeds; "
          f"excursions > 0.3 after Dice > 0.5: {exc}/{len(learned)} seeds; "
          f"Levene p = {stats.levene(f10, acc.heldout_dice, center='median').pvalue:.3g}")
else:
    tb = None
    print("literal batch 10 not available yet -> Figures 4 and 5 skipped")

# Figure 6: input distribution
from skimage.transform import resize as _rs
ids7 = idx_all.sort_values("img_id").iloc[::11]["image_path"][:24]
raw7 = np.concatenate([_rs(tifffile.imread(p).astype(np.float32), (256, 256), order=1, preserve_range=True,
                           anti_aliasing=True).ravel() for p in ids7])
x1 = raw7 / 255.0
x2 = (np.clip(raw7, -1350, 150) + 1350) / 1500
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(x1, bins=200, color="#c44e52"); ax[0].set_yscale("log")
ax[0].set(title="A  input after  image / 255", xlabel="network input value", ylabel="pixels")
ax[0].text(.97, .95, f"range [{x1.min():.1f}, {x1.max():.1f}]\nmean {x1.mean():.2f}  sign-changing",
           transform=ax[0].transAxes, ha="right", va="top", fontsize=8)
ax[1].hist(x2, bins=200, color="#4c72b0"); ax[1].set_yscale("log")
ax[1].set(title="B  input after  lung-window scaling", xlabel="network input value")
ax[1].text(.03, .95, "range [0, 1]\nbounded", transform=ax[1].transAxes, va="top", fontsize=8)
fig.suptitle("Distribution of network inputs (24 slices pooled)", fontsize=9)
psave(fig, "fig6_input_distribution")

# Figure 7: progressive protocol
fig, ax = plt.subplots(figsize=(6.5, 4))
lab8 = ["A\n/255, last-epoch", "B\n+best-epoch\n+tuned thr", "C\n+lung window", "D\n+augment"]
for i, a in enumerate(arms):
    v = gd.loc[gd.arm == a, "heldout_dice"].values
    ax.scatter(np.full(v.size, i) + RNG.uniform(-.12, .12, v.size), v, s=20, alpha=.8)
    ax.hlines(v.mean(), i - .25, i + .25, color="k", lw=2)
ax.set_xticks(range(4)); ax.set_xticklabels(lab8, fontsize=8)
ax.set(ylabel="held-out Dice @thr (12 seeds)", title="Progressive protocol (12 matched seeds per arm)")
psave(fig, "fig7_progressive_protocol")

# Figure 8: distribution of run outcomes across the study
groups9 = [("Baseline, accumulation (20)", acc.heldout_dice.values)]
if tb is not None:
    groups9.append((f"Baseline, literal batch 10 ({len(tb)})", tb.heldout_dice.values))
groups9 += [(f"Arm {a[0]} (12)", gd.loc[gd.arm == a, "heldout_dice"].values) for a in arms]
n9 = len(groups9)
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, (g, v) in enumerate(groups9):
    y = n9 - 1 - i
    ax.scatter(v, np.full(v.size, y) + RNG.uniform(-.15, .15, v.size), s=16, alpha=.75)
    ax.vlines(v.mean(), y - .3, y + .3, color="k", lw=2)
ax.set_yticks(range(n9)); ax.set_yticklabels([g for g, _ in groups9][::-1])
ax.set(xlabel="held-out Dice @0.5", title="Distribution of run outcomes across the study")
psave(fig, "fig8_run_outcomes")

# Figure 9: architecture comparison
order10 = ["unet", "attn_unet", "resunet", "resunetpp"]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.boxplot([wide[a].values for a in order10], flierprops=dict(ms=3))
gm10 = np.median(wide.values)
ax.axhspan(gm10 - TOST_MARGIN, gm10 + TOST_MARGIN, color="grey", alpha=.2, label="±0.02 margin around median (scale only)")
ax.set_xticks(range(1, 5)); ax.set_xticklabels(["unet", "attn unet", "resunet", "resunetpp"])
ttl10 = "Architecture comparison: per-image held-out Dice"
ax.set(ylabel="per-image Dice (mean over 5 seeds)", title=ttl10)
ax.legend(fontsize=7, loc="lower left")
psave(fig, "fig9_architectures")

# Figure 10: critical-difference diagram
rs = ranks.sort_values()
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.set_xlim(4.2, .8); ax.set_ylim(-2.3, 1.6); ax.axis("off")
ax.hlines(0, 1, 4, color="k", lw=1)
for t in range(1, 5):
    ax.vlines(t, -.06, .06, color="k"); ax.text(t, .15, str(t), ha="center", fontsize=8)
for j, (a, r) in enumerate(rs.items()):
    yy = -.35 - .45 * j                     # one label level per method: no overlap
    ax.plot(r, 0, "o", color=C[a]); ax.vlines(r, yy + .05, 0, color=C[a], lw=.8)
    ax.text(r, yy, f"{a.replace('_', ' ')}\n({r:.2f})", ha="center", va="top", fontsize=8,
            bbox=dict(fc="white", ec="none", pad=1), zorder=3)
ax.hlines(1.1, 1, 1 + cd, color="k", lw=3); ax.text(1 + cd / 2, 1.25, f"CD = {cd:.2f}", ha="center", fontsize=8)
bar_y = .45
for i in range(len(rs)):               # maximal groups whose rank range is within CD
    j = i
    while j + 1 < len(rs) and rs.iloc[j + 1] - rs.iloc[i] <= cd:
        j += 1
    if j > i and (i == 0 or rs.iloc[j] - rs.iloc[i - 1] > cd):
        ax.hlines(bar_y, rs.iloc[i] - .03, rs.iloc[j] + .03, color="k", lw=2.5); bar_y += .15
sig = pair_table[pair_table.nemenyi_sig].pair.tolist()
ax.set_title(f"Critical-difference diagram (Friedman+Nemenyi, n={N} images)"
             + (" — no pair exceeds CD" if not sig else " — exceeds CD: " + ", ".join(sig)), fontsize=9)
psave(fig, "fig10_critical_difference")

# Figure 11: residual per-image Dice vs covariates
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for a_, c, lbl in zip(ax, covs, ["lung area fraction", "mean lung HU", "|z_norm − 0.5| (within patient)"]):
    a_.scatter(img[c], img.dice, s=9, alpha=.5)
    bm = img.groupby(pd.qcut(img[c], 6), observed=True).agg(x=(c, "mean"), y=("dice", "mean"))
    a_.plot(bm.x, bm.y, "r-o", ms=3); a_.set(xlabel=lbl)
ax[0].set_ylabel("per-image Dice")
fig.suptitle("Residual per-image error vs image covariates (red = binned mean)", fontsize=9)
psave(fig, "fig11_failure_modes")

# Figure 12: qualitative (copied from the supplementary figure, if it was made)
import shutil
for ext in ("pdf", "png"):
    src_ = os.path.join(FIG, f"supp_qualitative.{ext}")
    if os.path.exists(src_):
        shutil.copy(src_, os.path.join(PF, f"fig12_qualitative.{ext}"))

# Figure 13: split sensitivity
sc_col = {"random_slice": "#4c72b0", "z_blocked": "#dd8452", "patient_lopo": "#55a868"}
sc_lab = {"random_slice": "random slice (6 seeds)", "z_blocked": "Z-blocked (6 seeds x 4 folds)", "patient_lopo": "leave-one-patient-out (10 folds)"}
fig, ax = plt.subplots(figsize=(6.5, 4))
for j, a in enumerate(["unet", "resunetpp"]):
    for i, sch in enumerate(["random_slice", "z_blocked", "patient_lopo"]):
        v = lp[(lp.arch == a) & (lp.scheme == sch)].heldout_dice.values
        x = j * 3.6 + i
        ax.scatter(np.full(v.size, x) + RNG.uniform(-.1, .1, v.size), v, s=16, alpha=.8,
                   color=sc_col[sch], label=sc_lab[sch] if j == 0 else None)
        ax.hlines(v.mean(), x - .25, x + .25, color="k", lw=2)
ax.set_xticks([1, 4.6]); ax.set_xticklabels(["unet", "resunetpp"])
ax.set(ylabel="held-out Dice @thr", title="Split sensitivity: random vs Z-blocked vs patient-level")
ax.legend(fontsize=7)
psave(fig, "fig13_split_sensitivity")
print("article figures written to", os.path.relpath(PF, BASE))


## H. Reproducibility check
The gradient-accumulation baseline and arm A use the same configuration and seeds; with deterministic training their per-seed results agree.

In [ ]:
P("\n\n########## H. Reproducibility check ##########")
fr_ = acc.set_index("seed").heldout_dice
ga_ = gd[gd.arm == "A_baseline"].set_index("seed").heldout_dice
common_ = fr_.index.intersection(ga_.index)
dd = (fr_.loc[common_] - ga_.loc[common_]).abs()
P(f"baseline (accumulation) vs arm A, seeds {list(common_)}: per-seed |difference| mean {dd.mean():.4f}, max {dd.max():.4f}")


## Save the report

In [ ]:
with open(f"{TAB}/analysis_report.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_lines) + "\n")
P("\n" + "="*78); P("wrote analysis_report.txt + table_*.csv")